# SeisLM: pretrained inference

Load a Hugging Face encoder, extract features from a real waveform, and save the results. Choose **base** for a quick first run or **large** for a larger model. These encoders return features, not phase picks.

## 1. Install

**Colab:** choose a **Python 3** runtime (GPU optional), then run all cells. Python 3.11–3.13 is supported. Restart only if the installation asks you to.

**Local Jupyter:** run `uv sync --locked --group notebook`, select that environment, and skip the installation cell.

In [ ]:
import importlib
import importlib.metadata as package_metadata
import subprocess
import sys

print("Python:", sys.version)
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "SeisLM 0.2.1 supports Python 3.11–3.13. "
        "This interpreter is outside that tested range; please report its version."
    )

# Detect loaded libraries that an installation would replace in this kernel.
_loaded_versions = {}
for _name in ("numpy", "matplotlib", "obspy", "seisLM", "torch"):
    if _name in sys.modules:
        try:
            _loaded_versions[_name] = package_metadata.version(_name)
        except package_metadata.PackageNotFoundError:
            pass

# Raise on installation failure so Run all cannot silently use old packages.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "https://huggingface.co/ltlltl2006/seislm-base/resolve/13a3dcd6ae56c39f5b0750b817c8788a7fc3c99b/seislm-0.2.1-py3-none-any.whl",
     "obspy==1.5.1", "matplotlib==3.11.2"],
    check=True,
)

importlib.invalidate_caches()
_changed = [
    name for name, previous in _loaded_versions.items()
    if package_metadata.version(name) != previous
]
if _changed:
    raise RuntimeError(
        "Installed new versions of loaded libraries: " + ", ".join(_changed)
        + ". Restart the kernel once, then run the notebook from the top."
    )


## 2. Choose a model and device

Set `MODEL_NAME` to `"base"` or `"large"`. `DEVICE="auto"` uses a GPU when available. Change `OUTPUT_DIR` to choose where results are saved.

In [ ]:
import os
from pathlib import Path

MODEL_NAME = os.environ.get("SEISLM_NOTEBOOK_MODEL", "base")  # "base" or "large"
DEVICE = os.environ.get("SEISLM_NOTEBOOK_DEVICE", "auto")  # "auto", "cpu", "cuda"
OUTPUT_DIR = Path(os.environ.get("SEISLM_NOTEBOOK_OUTPUT_DIR", "seislm_inference_outputs")).expanduser().resolve()
CACHE_DIR = os.environ.get("SEISLM_NOTEBOOK_CACHE_DIR") or None

RELEASES = {
    "base": {
        "repo_id": "ltlltl2006/seislm-base",
        "revision": "a9e9f79af940a3ea28a49ef9fa495ae89b48d576",
        "hidden_size": 240,
    },
    "large": {
        "repo_id": "ltlltl2006/seislm-large",
        "revision": "3ad2a723faeded0ca45b555c9c0ea8823affcc4f",
        "hidden_size": 768,
    },
}
if MODEL_NAME not in RELEASES or DEVICE not in {"auto", "cpu", "cuda"}:
    raise ValueError("Choose model base/large and device auto/cpu/cuda.")
release = RELEASES[MODEL_NAME]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import importlib.metadata
import json
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import obspy
import torch
import seisLM
from seisLM import encode, load_pretrained

%matplotlib inline

started = time.perf_counter()
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but is unavailable. Select a GPU runtime or set DEVICE='cpu'.")
device = "cuda" if DEVICE == "auto" and torch.cuda.is_available() else DEVICE
if device == "auto":
    device = "cpu"
versions = {
    "python": platform.python_version(),
    **{name: importlib.metadata.version(name) for name in ("torch", "numpy", "obspy", "matplotlib", "seisLM")},
}
print(json.dumps(versions, indent=2))
print("Device:", device, torch.cuda.get_device_name() if device == "cuda" else platform.machine())
print("Model:", release["repo_id"])
print("Revision:", release["revision"])
print("Output folder:", OUTPUT_DIR)


## 3. Load a waveform

ObsPy includes this 30-second recording from station **BW.RJOB**: three components at **100 Hz**, ordered **Z, N, E**.

For your own waveform, replace `obspy.read()` and adjust the example shape check. Supply aligned, finite ZNE components at 100 Hz; handle gaps, orientation, and resampling before inference.

In [ ]:
stream = obspy.read()
traces = []
for component in "ZNE":
    selected = stream.select(component=component)
    if len(selected) != 1:
        raise ValueError(f"Expected exactly one {component} component.")
    traces.append(selected[0])

sampling_rate = float(traces[0].stats.sampling_rate)
start_time = traces[0].stats.starttime
sample_count = traces[0].stats.npts
if sampling_rate != 100.0:
    raise ValueError("The pretrained models expect 100 Hz inputs.")
for trace in traces:
    if (trace.stats.sampling_rate != sampling_rate or trace.stats.npts != sample_count
            or trace.stats.starttime != start_time):
        raise ValueError("Components must have matching sampling rates, lengths, and start times.")
    if np.ma.isMaskedArray(trace.data) and np.ma.getmaskarray(trace.data).any():
        raise ValueError("Waveform contains gaps; handle them explicitly before inference.")
raw_waveforms = np.stack([trace.data for trace in traces]).astype(np.float32)
assert raw_waveforms.shape == (3, 3000) and np.isfinite(raw_waveforms).all()
sample_times = np.arange(sample_count) / sampling_rate
print(stream)
print("Selected component order: ZNE")
print("Input array:", raw_waveforms.shape, raw_waveforms.dtype)


## 4. Normalize

Demean each component and divide by its population standard deviation. The model input has shape **[batch, component, sample]** and dtype float32.

In [ ]:
waveforms = torch.from_numpy(raw_waveforms.copy()).unsqueeze(0)
waveforms = waveforms - waveforms.mean(dim=-1, keepdim=True)
std = waveforms.std(dim=-1, correction=0, keepdim=True)
std = torch.where(std == 0, torch.ones_like(std), std)
waveforms = waveforms / (std + 1e-10)
assert waveforms.dtype == torch.float32 and torch.isfinite(waveforms).all()

fig, axes = plt.subplots(3, 2, figsize=(12, 6), sharex=True, constrained_layout=True)
for i, component in enumerate("ZNE"):
    axes[i, 0].plot(sample_times, raw_waveforms[i], linewidth=0.6)
    axes[i, 1].plot(sample_times, waveforms[0, i].numpy(), linewidth=0.6)
    axes[i, 0].set_ylabel(component)
axes[0, 0].set_title("Example amplitude as provided")
axes[0, 1].set_title("Per-component normalized amplitude")
for ax in axes[-1]:
    ax.set_xlabel("Seconds since waveform start")
fig.savefig(OUTPUT_DIR / "waveforms.png", dpi=140)
plt.show()


## 5. Extract features

The model downloads from a pinned public Hugging Face release and is cached for later runs. This waveform produces **749 frames**, with **240** features for base or **768** for large.

In [ ]:
model = load_pretrained(
    release["repo_id"], revision=release["revision"], device=device, cache_dir=CACHE_DIR
)
if device == "cuda":
    torch.cuda.synchronize()
inference_started = time.perf_counter()
features = encode(model, waveforms)
if device == "cuda":
    torch.cuda.synchronize()
inference_seconds = time.perf_counter() - inference_started
features_array = features.cpu().numpy()
token_count = waveforms.shape[-1]
for kernel, stride in zip(model.config.conv_kernel, model.config.conv_stride):
    token_count = (token_count - kernel) // stride + 1
expected_shape = (waveforms.shape[0], token_count, release["hidden_size"])
assert features_array.shape == expected_shape
assert features_array.dtype == np.float32 and np.isfinite(features_array).all()
print("Input:", tuple(waveforms.shape))
print("Features:", features_array.shape)
print(f"Encoder execution: {inference_seconds:.3f} s (excludes model download/loading)")


## 6. Compare the input and features

Both panels share a time axis. Each feature frame is placed at the center of its convolutional input window; its values incorporate context from the whole recording.

In [ ]:
# The normalized waveform actually passed to the model.
input_array = waveforms[0].detach().cpu().numpy()
sample_times = np.arange(input_array.shape[-1]) / sampling_rate

# Position feature frames at their convolutional receptive-field centers.
stride_samples = 1
center_samples = 0.0
for kernel, stride in zip(model.config.conv_kernel, model.config.conv_stride):
    center_samples += (kernel - 1) * stride_samples / 2
    stride_samples *= stride

token_times = (
    center_samples + stride_samples * np.arange(features_array.shape[1])
) / sampling_rate
half_step = stride_samples / sampling_rate / 2
limit = float(np.quantile(np.abs(features_array), 0.99)) or 1.0

fig, axes = plt.subplots(
    2, 1,
    figsize=(12, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 2]},
    constrained_layout=True,
)
ax_waveform, ax_features = axes

for component, trace in zip("ZNE", input_array):
    ax_waveform.plot(
        sample_times, trace, label=component, linewidth=0.7, alpha=0.8
    )

ax_waveform.set_title("Normalized model input")
ax_waveform.set_ylabel("Amplitude")
ax_waveform.legend(loc="upper right", ncol=3)

image = ax_features.imshow(
    features_array[0].T,
    origin="lower",
    aspect="auto",
    cmap="RdBu_r",
    vmin=-limit,
    vmax=limit,
    extent=(
        token_times[0] - half_step,
        token_times[-1] + half_step,
        -0.5,
        features_array.shape[2] - 0.5,
    ),
)
ax_features.set_title(
    f"SeisLM {MODEL_NAME}: "
    f"{features_array.shape[1]} frames × "
    f"{features_array.shape[2]} feature dimensions"
)
ax_features.set_ylabel("Feature dimension")
ax_features.set_xlabel("Time since waveform start [s]")
ax_features.set_xlim(0, input_array.shape[-1] / sampling_rate)

# Keep both panels the same width when adding the colorbar.
fig.colorbar(image, ax=axes, label="Feature value", shrink=0.8)

fig.savefig(OUTPUT_DIR / "features.png", dpi=140)
plt.show()

## 7. Save the results

Save the features, frame times, plots, and model/preprocessing metadata. In Colab, download them from `seislm_inference_outputs` in the **Files** pane. Reload features with `np.load("contextual_features.npy", allow_pickle=False)`.

In [ ]:
np.save(OUTPUT_DIR / "contextual_features.npy", features_array, allow_pickle=False)
np.save(OUTPUT_DIR / "token_times.npy", token_times, allow_pickle=False)
summary = {
    "model": {"name": MODEL_NAME, **release},
    "versions": versions,
    "device": str(next(model.parameters()).device),
    "device_name": torch.cuda.get_device_name() if device == "cuda" else platform.machine(),
    "waveform": {
        "source": "ObsPy bundled example (obspy.read())",
        "trace_ids": [trace.id for trace in traces],
        "start_time_utc": str(start_time),
        "sampling_rate_hz": sampling_rate,
        "component_order": "ZNE",
        "samples_per_component": int(sample_count),
    },
    "preprocessing": {"demean": "per component", "std_correction": 0, "epsilon": 1e-10, "resampling": False, "gap_filling": False},
    "input_shape": list(waveforms.shape),
    "feature_shape": list(features_array.shape),
    "dtype": str(features_array.dtype),
    "finite_features": bool(np.isfinite(features_array).all()),
    "inference_seconds": inference_seconds,
    "elapsed_seconds": time.perf_counter() - started,
    "artifacts": ["contextual_features.npy", "token_times.npy", "waveforms.png", "features.png"],
}
(OUTPUT_DIR / "run-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
assert json.loads((OUTPUT_DIR / "run-summary.json").read_text()) == summary
np.testing.assert_array_equal(np.load(OUTPUT_DIR / "contextual_features.npy", allow_pickle=False), features_array)
for filename in summary["artifacts"]:
    assert (OUTPUT_DIR / filename).is_file()
print(json.dumps(summary, indent=2))
print("Saved outputs to:", OUTPUT_DIR)
